# Lekh Neural CUDA Training

This notebook trains one checksum-pinned Lekh candidate on a Colab GPU. The Mac does no PyTorch training. Every completed epoch is mirrored to your private Google Drive folder, and the returned checkpoint remains unpromoted until macOS Core ML conversion, parity, quality, and device gates pass.

**Before running:** choose **Runtime → Change runtime type → GPU**. Free Colab GPU availability and runtime duration are not guaranteed.

## 1. Mount Drive and authenticate the exact bundle

Expected bundle: `lekh-neural-lekh-open-vocab-ctc-transformer-v2-cuda-training-abc8ecfb2bfbcf32.tar.gz`  
SHA-256: `b5968bad47dbeda072e213ee9e649ba5d14645f62e938a4524fae977d6684628`

The cell first reuses an exact local runtime copy, then an exact durable Drive copy. It asks for a browser upload only when neither exists. Every path is checked by byte count and SHA-256 before extraction.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import re
import shutil
import sys
import uuid

EXPECTED_ARCHIVE_NAME = 'lekh-neural-lekh-open-vocab-ctc-transformer-v2-cuda-training-abc8ecfb2bfbcf32.tar.gz'
EXPECTED_ARCHIVE_SHA256 = 'b5968bad47dbeda072e213ee9e649ba5d14645f62e938a4524fae977d6684628'
EXPECTED_ARCHIVE_BYTES = 99316415
EXPECTED_BUNDLE_ID = 'abc8ecfb2bfbcf3201cc2ad741b8c7ca98714882d25e93a0c38b900b3f136296'
EXPECTED_MODEL_ID = 'lekh-open-vocab-ctc-transformer-v2'
EXPECTED_CONFIG = 'data/neural/training/open-vocab-ctc-transformer-v2.config.json'
EXPECTED_CANDIDATE_PREFIX = 'data/generated/neural-open-vocab-model/lekh-open-vocab-ctc-transformer-v2'
VERIFIER_MODULE_SOURCE = '#!/usr/bin/env python3\n"""Closed-inventory archives for split-host neural training.\n\nThe module intentionally depends only on the Python standard library so the\nsame verifier can run before any third-party package is installed in Colab.\n"""\n\nfrom __future__ import annotations\n\nimport gzip\nimport hashlib\nimport json\nimport os\nimport re\nimport shutil\nimport stat\nimport tarfile\nimport tempfile\nimport uuid\nfrom dataclasses import dataclass\nfrom pathlib import Path, PurePosixPath\nfrom typing import Any, BinaryIO, Iterable\n\n\nSHA256_LENGTH = 64\nCHUNK_BYTES = 1024 * 1024\nMAX_ARCHIVE_BYTES = 2 * 1024 * 1024 * 1024\nMAX_MEMBER_BYTES = 1024 * 1024 * 1024\nMAX_TOTAL_BYTES = 2 * 1024 * 1024 * 1024\nMAX_FILES = 96\nMAX_MANIFEST_BYTES = 4 * 1024 * 1024\nMAX_FILENAME_COMPONENT_BYTES = 180\n\nSAFE_FILENAME_COMPONENT = re.compile(r"^[A-Za-z0-9][A-Za-z0-9._-]*$")\nSAFE_ROLE = re.compile(r"^[a-z0-9][a-z0-9-]*$")\n\nBUNDLE_KIND = "lekh-neural-remote-training-bundle-v1"\nRESULT_KIND = "lekh-neural-remote-training-result-v1"\n\nARCHIVE_POLICIES = {\n    BUNDLE_KIND: {\n        "prefix": "lekh-neural-remote",\n        "manifest": "NEURAL_REMOTE_BUNDLE_MANIFEST.json",\n        "identity": "bundleId",\n    },\n    RESULT_KIND: {\n        "prefix": "lekh-neural-result",\n        "manifest": "NEURAL_REMOTE_RESULT_MANIFEST.json",\n        "identity": "resultId",\n    },\n}\n\nSEQ2SEQ_CONFIG = (\n    "data/neural/training/open-vocab-seq2seq-v1.config.json"\n)\nBIGRU_ATTENTION_CONFIG = (\n    "data/neural/training/open-vocab-bigru-attention-v1.config.json"\n)\nCTC_TRANSFORMER_CONFIG = (\n    "data/neural/training/open-vocab-ctc-transformer-v2.config.json"\n)\nSEQ2SEQ_TRAINER = "scripts/train-open-vocab-seq2seq-transliterator.py"\nCTC_TRANSFORMER_TRAINER = (\n    "scripts/train-open-vocab-ctc-transformer.py"\n)\nCTC_TRANSFORMER_SHARED_MODEL = (\n    "scripts/lib/neural_ctc_transformer.py"\n)\n\nTRAINER_BY_CONFIG = {\n    SEQ2SEQ_CONFIG: SEQ2SEQ_TRAINER,\n    BIGRU_ATTENTION_CONFIG: SEQ2SEQ_TRAINER,\n    CTC_TRANSFORMER_CONFIG: CTC_TRANSFORMER_TRAINER,\n}\nSUPPORTED_CONFIGS = frozenset(TRAINER_BY_CONFIG)\n\nTRAINER_RUNTIME_FILES_BY_CONFIG = {\n    SEQ2SEQ_CONFIG: (\n        (SEQ2SEQ_TRAINER, "trainer"),\n    ),\n    BIGRU_ATTENTION_CONFIG: (\n        (SEQ2SEQ_TRAINER, "trainer"),\n    ),\n    CTC_TRANSFORMER_CONFIG: (\n        (CTC_TRANSFORMER_TRAINER, "trainer"),\n        (SEQ2SEQ_TRAINER, "trainer-dependency"),\n        (CTC_TRANSFORMER_SHARED_MODEL, "trainer-dependency"),\n    ),\n}\n\nBUNDLE_RUNTIME_FILES = (\n    (\n        "scripts/check-neural-open-vocab-toolchain.py",\n        "toolchain-verifier",\n    ),\n    (\n        "scripts/run-neural-remote-training.py",\n        "remote-runner",\n    ),\n    (\n        "scripts/verify-neural-remote-training-bundle.py",\n        "bundle-verifier",\n    ),\n    (\n        "scripts/lib/neural_remote_artifacts.py",\n        "archive-contract",\n    ),\n    (\n        "requirements/neural-open-vocab.lock",\n        "python-requirements",\n    ),\n    (\n        "requirements/neural-open-vocab-cu118.lock",\n        "cuda-python-requirements",\n    ),\n)\n\n\nclass NeuralRemoteArtifactError(RuntimeError):\n    """Raised when an archive or its source inventory fails closed."""\n\n\ndef trainer_path_for_config(config_relative_path: str) -> str:\n    config_relative = safe_relative_path(\n        config_relative_path,\n        "training config path",\n    )\n    trainer_path = TRAINER_BY_CONFIG.get(config_relative)\n    if trainer_path is None:\n        raise NeuralRemoteArtifactError(\n            f"Unsupported remote training config: {config_relative}"\n        )\n    return trainer_path\n\n\ndef trainer_runtime_files_for_config(\n    config_relative_path: str,\n) -> tuple[tuple[str, str], ...]:\n    config_relative = safe_relative_path(\n        config_relative_path,\n        "training config path",\n    )\n    files = TRAINER_RUNTIME_FILES_BY_CONFIG.get(config_relative)\n    if files is None:\n        raise NeuralRemoteArtifactError(\n            f"Unsupported remote training config: {config_relative}"\n        )\n    return files\n\n\n@dataclass(frozen=True)\nclass ArchiveFile:\n    source: Path\n    archive_path: str\n    role: str\n    expected_sha256: str | None = None\n    expected_bytes: int | None = None\n\n\nclass DigestingReader:\n    """A minimal non-seekable reader that records exact bytes consumed."""\n\n    def __init__(self, handle: BinaryIO):\n        self.handle = handle\n        self.digest = hashlib.sha256()\n        self.bytes_read = 0\n\n    def read(self, size: int = -1) -> bytes:\n        value = self.handle.read(size)\n        self.digest.update(value)\n        self.bytes_read += len(value)\n        return value\n\n    def hexdigest(self) -> str:\n        return self.digest.hexdigest()\n\n\ndef canonical_json_bytes(value: Any) -> bytes:\n    return json.dumps(\n        value,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(",", ":"),\n    ).encode("utf-8")\n\n\ndef sha256_bytes(value: bytes) -> str:\n    return hashlib.sha256(value).hexdigest()\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open_regular_binary(path) as handle:\n        metadata_before = os.fstat(handle.fileno())\n        for chunk in iter(lambda: handle.read(CHUNK_BYTES), b""):\n            digest.update(chunk)\n        metadata_after = os.fstat(handle.fileno())\n    metadata_path = path.lstat()\n    if (\n        metadata_after.st_dev != metadata_before.st_dev\n        or metadata_after.st_ino != metadata_before.st_ino\n        or metadata_after.st_size != metadata_before.st_size\n        or metadata_after.st_mtime_ns != metadata_before.st_mtime_ns\n        or metadata_path.st_dev != metadata_before.st_dev\n        or metadata_path.st_ino != metadata_before.st_ino\n        or metadata_path.st_size != metadata_before.st_size\n        or metadata_path.st_mtime_ns != metadata_before.st_mtime_ns\n    ):\n        raise NeuralRemoteArtifactError(\n            f"Regular file changed while hashing: {path}"\n        )\n    return digest.hexdigest()\n\n\ndef is_sha256(value: Any) -> bool:\n    return (\n        isinstance(value, str)\n        and len(value) == SHA256_LENGTH\n        and all(character in "0123456789abcdef" for character in value)\n    )\n\n\ndef safe_relative_path(value: str, label: str = "path") -> str:\n    if not isinstance(value, str) or not value or "\\\\" in value:\n        raise NeuralRemoteArtifactError(f"{label} is not a portable path.")\n    parsed = PurePosixPath(value)\n    if parsed.is_absolute() or any(\n        component in {"", ".", ".."} for component in parsed.parts\n    ):\n        raise NeuralRemoteArtifactError(f"{label} escapes its archive root.")\n    normalized = parsed.as_posix()\n    if normalized != value or len(normalized.encode("utf-8")) > 240:\n        raise NeuralRemoteArtifactError(f"{label} is non-canonical or too long.")\n    return normalized\n\n\ndef safe_filename_component(value: str, label: str = "filename") -> str:\n    if (\n        not isinstance(value, str)\n        or not SAFE_FILENAME_COMPONENT.fullmatch(value)\n        or len(value.encode("utf-8")) > MAX_FILENAME_COMPONENT_BYTES\n        or value in {".", ".."}\n    ):\n        raise NeuralRemoteArtifactError(\n            f"{label} is not a safe portable filename component."\n        )\n    return value\n\n\ndef contained_regular_file(root: Path, relative_path: str) -> Path:\n    relative = safe_relative_path(relative_path)\n    requested_root = root.absolute()\n    try:\n        root_metadata = requested_root.lstat()\n    except FileNotFoundError as error:\n        raise NeuralRemoteArtifactError(\n            f"Required file root is missing: {requested_root}"\n        ) from error\n    if (\n        stat.S_ISLNK(root_metadata.st_mode)\n        or not stat.S_ISDIR(root_metadata.st_mode)\n    ):\n        raise NeuralRemoteArtifactError(\n            f"Required file root is unsafe: {requested_root}"\n        )\n    resolved_root = requested_root.resolve(strict=True)\n    candidate = resolved_root.joinpath(*PurePosixPath(relative).parts)\n    try:\n        metadata = candidate.lstat()\n    except FileNotFoundError as error:\n        raise NeuralRemoteArtifactError(\n            f"Required bundle input is missing: {relative}"\n        ) from error\n    if stat.S_ISLNK(metadata.st_mode) or not stat.S_ISREG(metadata.st_mode):\n        raise NeuralRemoteArtifactError(\n            f"Bundle input is not a regular non-symlink file: {relative}"\n        )\n    resolved = candidate.resolve(strict=True)\n    if not resolved.is_relative_to(resolved_root):\n        raise NeuralRemoteArtifactError(\n            f"Bundle input escapes the repository: {relative}"\n        )\n    return resolved\n\n\ndef open_regular_binary(path: Path) -> BinaryIO:\n    try:\n        metadata = path.lstat()\n    except FileNotFoundError as error:\n        raise NeuralRemoteArtifactError(f"Missing regular file: {path}") from error\n    if stat.S_ISLNK(metadata.st_mode) or not stat.S_ISREG(metadata.st_mode):\n        raise NeuralRemoteArtifactError(\n            f"Refusing non-regular or symbolic-link file: {path}"\n        )\n    flags = os.O_RDONLY | getattr(os, "O_NOFOLLOW", 0)\n    descriptor = os.open(path, flags)\n    opened = os.fstat(descriptor)\n    if (\n        not stat.S_ISREG(opened.st_mode)\n        or opened.st_dev != metadata.st_dev\n        or opened.st_ino != metadata.st_ino\n    ):\n        os.close(descriptor)\n        raise NeuralRemoteArtifactError(\n            f"Regular file changed before it could be opened: {path}"\n        )\n    return os.fdopen(descriptor, "rb")\n\n\ndef read_json_object(path: Path, maximum_bytes: int = MAX_MANIFEST_BYTES) -> dict[str, Any]:\n    with open_regular_binary(path) as handle:\n        payload = handle.read(maximum_bytes + 1)\n    if len(payload) > maximum_bytes:\n        raise NeuralRemoteArtifactError(f"JSON artifact is too large: {path}")\n    try:\n        parsed = json.loads(payload.decode("utf-8"))\n    except (UnicodeDecodeError, json.JSONDecodeError) as error:\n        raise NeuralRemoteArtifactError(f"Invalid UTF-8 JSON: {path}") from error\n    if not isinstance(parsed, dict):\n        raise NeuralRemoteArtifactError(f"JSON artifact must be an object: {path}")\n    return parsed\n\n\ndef collect_training_bundle(\n    repo_root: Path,\n    config_relative_path: str,\n) -> tuple[dict[str, Any], list[ArchiveFile]]:\n    root = repo_root.resolve(strict=True)\n    config_relative = safe_relative_path(\n        config_relative_path,\n        "training config path",\n    )\n    trainer_relative = trainer_path_for_config(config_relative)\n    config_path = contained_regular_file(root, config_relative)\n    config = read_json_object(config_path)\n    model_id = config.get("modelId")\n    if not isinstance(model_id, str) or not model_id:\n        raise NeuralRemoteArtifactError("Training config lacks modelId.")\n\n    files: dict[str, ArchiveFile] = {}\n\n    def add(\n        relative_path: str,\n        role: str,\n        *,\n        expected_sha256: str | None = None,\n        expected_bytes: int | None = None,\n    ) -> None:\n        relative = safe_relative_path(relative_path)\n        if relative in files:\n            raise NeuralRemoteArtifactError(\n                f"Duplicate remote bundle path: {relative}"\n            )\n        if expected_sha256 is not None and not is_sha256(expected_sha256):\n            raise NeuralRemoteArtifactError(\n                f"Invalid expected digest for {relative}."\n            )\n        if expected_bytes is not None and (\n            type(expected_bytes) is not int or expected_bytes < 1\n        ):\n            raise NeuralRemoteArtifactError(\n                f"Invalid expected byte count for {relative}."\n            )\n        files[relative] = ArchiveFile(\n            source=contained_regular_file(root, relative),\n            archive_path=relative,\n            role=role,\n            expected_sha256=expected_sha256,\n            expected_bytes=expected_bytes,\n        )\n\n    for relative, role in (\n        *BUNDLE_RUNTIME_FILES,\n        *trainer_runtime_files_for_config(config_relative),\n    ):\n        add(relative, role)\n    add(config_relative, "training-config")\n\n    training = config.get("training")\n    evaluation = config.get("evaluation")\n    if not isinstance(training, dict) or not isinstance(evaluation, dict):\n        raise NeuralRemoteArtifactError(\n            "Training config lacks training/evaluation sections."\n        )\n    dataset_relative = safe_relative_path(\n        training.get("datasetManifest"),\n        "dataset manifest path",\n    )\n    gold_relative = safe_relative_path(\n        evaluation.get("goldManifest"),\n        "gold manifest path",\n    )\n    official_relative = safe_relative_path(\n        evaluation.get("officialBenchmarkManifest"),\n        "official benchmark manifest path",\n    )\n\n    dataset_path = contained_regular_file(root, dataset_relative)\n    dataset = read_json_object(dataset_path)\n    add(dataset_relative, "dataset-manifest")\n    for split in ("train", "dev", "test"):\n        try:\n            split_path = dataset["splitFiles"][split]\n            split_sha256 = dataset["sha256"][split]\n            split_bytes = dataset["bytes"][split]\n        except (KeyError, TypeError) as error:\n            raise NeuralRemoteArtifactError(\n                f"Dataset manifest lacks {split} inventory."\n            ) from error\n        add(\n            split_path,\n            f"dataset-{split}",\n            expected_sha256=split_sha256,\n            expected_bytes=split_bytes,\n        )\n\n    gold_path = contained_regular_file(root, gold_relative)\n    gold = read_json_object(gold_path)\n    add(gold_relative, "gold-manifest")\n    add_manifest_suites(add, gold, "gold-suite")\n\n    official_path = contained_regular_file(root, official_relative)\n    official = read_json_object(official_path)\n    add(official_relative, "official-benchmark-manifest")\n    add_manifest_suites(add, official, "official-benchmark-suite")\n\n    dataset_identity = dataset.get("datasetContentSha256")\n    gold_identity = gold.get("corpusSha256")\n    official_identity = official.get("corpusSha256")\n    if not all(\n        is_sha256(value)\n        for value in (dataset_identity, gold_identity, official_identity)\n    ):\n        raise NeuralRemoteArtifactError(\n            "Dataset/gold/benchmark corpus identity is invalid."\n        )\n\n    manifest_base = {\n        "schemaVersion": 1,\n        "modelId": model_id,\n        "trainingConfig": config_relative,\n        "trainerPath": trainer_relative,\n        "datasetManifest": dataset_relative,\n        "datasetContentSha256": dataset_identity,\n        "goldManifest": gold_relative,\n        "goldCorpusSha256": gold_identity,\n        "officialBenchmarkManifest": official_relative,\n        "officialBenchmarkCorpusSha256": official_identity,\n        "trainingProtocol": {\n            "device": "cuda",\n            "skipTrain": False,\n            "skipCoreML": True,\n            "deterministicAlgorithms": True,\n            "cublasWorkspaceConfig": ":4096:8",\n        },\n    }\n    return manifest_base, sorted(\n        files.values(),\n        key=lambda item: item.archive_path,\n    )\n\n\ndef add_manifest_suites(\n    add: Any,\n    manifest: dict[str, Any],\n    role: str,\n) -> None:\n    suites = manifest.get("suites")\n    if not isinstance(suites, list) or not suites:\n        raise NeuralRemoteArtifactError(f"{role} manifest has no suites.")\n    for suite in suites:\n        if not isinstance(suite, dict):\n            raise NeuralRemoteArtifactError(f"{role} inventory is invalid.")\n        add(\n            suite.get("path"),\n            role,\n            expected_sha256=suite.get("sha256"),\n        )\n\n\ndef manifest_identity(manifest: dict[str, Any], identity_field: str) -> str:\n    unsigned = dict(manifest)\n    unsigned.pop(identity_field, None)\n    return sha256_bytes(canonical_json_bytes(unsigned))\n\n\ndef build_closed_archive(\n    *,\n    source_root: Path,\n    output_dir: Path,\n    artifact_kind: str,\n    filename_stem: str,\n    manifest_base: dict[str, Any],\n    files: Iterable[ArchiveFile],\n    compression_level: int = 1,\n) -> dict[str, Any]:\n    policy = archive_policy(artifact_kind)\n    if type(compression_level) is not int or not 0 <= compression_level <= 9:\n        raise NeuralRemoteArtifactError("Gzip compression level must be 0-9.")\n    filename_stem = safe_filename_component(\n        filename_stem,\n        "archive filename stem",\n    )\n    if (\n        not isinstance(manifest_base, dict)\n        or manifest_base.get("schemaVersion") != 1\n    ):\n        raise NeuralRemoteArtifactError(\n            "Archive manifest base must use schemaVersion 1."\n        )\n    reserved_manifest_fields = {\n        "artifactKind",\n        "archivePrefix",\n        "compression",\n        "files",\n        policy["identity"],\n    }\n    if reserved_manifest_fields.intersection(manifest_base):\n        raise NeuralRemoteArtifactError(\n            "Archive manifest base contains a reserved contract field."\n        )\n\n    requested_root = source_root.absolute()\n    root_metadata = requested_root.lstat()\n    if (\n        stat.S_ISLNK(root_metadata.st_mode)\n        or not stat.S_ISDIR(root_metadata.st_mode)\n    ):\n        raise NeuralRemoteArtifactError(\n            f"Archive source root is unsafe: {requested_root}"\n        )\n    root = requested_root.resolve(strict=True)\n    requested_output_dir = output_dir.absolute()\n    if requested_output_dir.is_symlink():\n        raise NeuralRemoteArtifactError(\n            f"Archive output directory is unsafe: {requested_output_dir}"\n        )\n    requested_output_dir.mkdir(parents=True, exist_ok=True)\n    destination_dir = requested_output_dir.resolve(strict=True)\n    if not destination_dir.is_dir():\n        raise NeuralRemoteArtifactError(\n            f"Archive output directory is unsafe: {destination_dir}"\n        )\n    specs = sorted(files, key=lambda item: item.archive_path)\n    if not specs or len(specs) > MAX_FILES:\n        raise NeuralRemoteArtifactError("Archive file inventory is empty or too large.")\n    if len({item.archive_path for item in specs}) != len(specs):\n        raise NeuralRemoteArtifactError("Archive file inventory contains duplicates.")\n\n    staging = destination_dir / (\n        f".{filename_stem}.staging.{os.getpid()}.{uuid.uuid4().hex}.tar.gz"\n    )\n    entries: list[dict[str, Any]] = []\n    total_bytes = 0\n    manifest: dict[str, Any] | None = None\n    manifest_bytes: bytes | None = None\n    try:\n        with staging.open("xb") as raw:\n            with gzip.GzipFile(\n                filename="",\n                mode="wb",\n                compresslevel=compression_level,\n                fileobj=raw,\n                mtime=0,\n            ) as compressed:\n                with tarfile.open(\n                    fileobj=compressed,\n                    mode="w|",\n                    format=tarfile.USTAR_FORMAT,\n                ) as archive:\n                    for spec in specs:\n                        relative = safe_relative_path(\n                            spec.archive_path,\n                            "archive member path",\n                        )\n                        if (\n                            not isinstance(spec.role, str)\n                            or not SAFE_ROLE.fullmatch(spec.role)\n                            or len(spec.role) > 80\n                        ):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive member role is invalid: {relative}"\n                            )\n                        requested_source = spec.source.absolute()\n                        requested_metadata = requested_source.lstat()\n                        if stat.S_ISLNK(requested_metadata.st_mode):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive source is a symbolic link: {relative}"\n                            )\n                        source = requested_source.resolve(strict=True)\n                        if not source.is_relative_to(root):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive source escapes its root: {source}"\n                            )\n                        metadata_before = source.lstat()\n                        if (\n                            stat.S_ISLNK(metadata_before.st_mode)\n                            or not stat.S_ISREG(metadata_before.st_mode)\n                            or metadata_before.st_size < 1\n                            or metadata_before.st_size > MAX_MEMBER_BYTES\n                        ):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive source is unsafe: {relative}"\n                            )\n                        member = regular_tar_info(\n                            f"{policy[\'prefix\']}/{relative}",\n                            metadata_before.st_size,\n                        )\n                        with open_regular_binary(source) as handle:\n                            digesting = DigestingReader(handle)\n                            archive.addfile(member, digesting)\n                        metadata_after = source.lstat()\n                        if (\n                            digesting.bytes_read != metadata_before.st_size\n                            or metadata_after.st_dev != metadata_before.st_dev\n                            or metadata_after.st_ino != metadata_before.st_ino\n                            or metadata_after.st_size != metadata_before.st_size\n                            or metadata_after.st_mtime_ns\n                                != metadata_before.st_mtime_ns\n                        ):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive source changed while reading: {relative}"\n                            )\n                        observed_sha256 = digesting.hexdigest()\n                        if (\n                            spec.expected_sha256 is not None\n                            and observed_sha256 != spec.expected_sha256\n                        ):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive source digest is stale: {relative}"\n                            )\n                        if (\n                            spec.expected_bytes is not None\n                            and metadata_before.st_size != spec.expected_bytes\n                        ):\n                            raise NeuralRemoteArtifactError(\n                                f"Archive source byte count is stale: {relative}"\n                            )\n                        total_bytes += metadata_before.st_size\n                        if total_bytes > MAX_TOTAL_BYTES:\n                            raise NeuralRemoteArtifactError(\n                                "Archive exceeds the uncompressed safety limit."\n                            )\n                        entries.append(\n                            {\n                                "path": relative,\n                                "role": spec.role,\n                                "sha256": observed_sha256,\n                                "bytes": metadata_before.st_size,\n                            }\n                        )\n\n                    unsigned_manifest = {\n                        **manifest_base,\n                        "artifactKind": artifact_kind,\n                        "archivePrefix": policy["prefix"],\n                        "compression": "tar-gzip-deterministic-v1",\n                        "files": entries,\n                    }\n                    identity = manifest_identity(\n                        unsigned_manifest,\n                        policy["identity"],\n                    )\n                    manifest = {\n                        **unsigned_manifest,\n                        policy["identity"]: identity,\n                    }\n                    manifest_bytes = canonical_json_bytes(manifest) + b"\\n"\n                    archive.addfile(\n                        regular_tar_info(\n                            f"{policy[\'prefix\']}/{policy[\'manifest\']}",\n                            len(manifest_bytes),\n                        ),\n                        BytesReader(manifest_bytes),\n                    )\n            raw.flush()\n            os.fsync(raw.fileno())\n\n        if manifest is None or manifest_bytes is None:\n            raise NeuralRemoteArtifactError("Archive manifest was not created.")\n        identity = manifest[policy["identity"]]\n        target = destination_dir / (\n            f"{filename_stem}-{identity[:16]}.tar.gz"\n        )\n        archive_sha256 = sha256_file(staging)\n        if target.exists() or target.is_symlink():\n            if (\n                target.is_symlink()\n                or not target.is_file()\n                or sha256_file(target) != archive_sha256\n            ):\n                raise NeuralRemoteArtifactError(\n                    f"Archive target already exists with different bytes: {target}"\n                )\n            staging.unlink()\n        else:\n            os.replace(staging, target)\n        return {\n            "schemaVersion": 1,\n            "status": "passed-closed-archive-build",\n            "artifactKind": artifact_kind,\n            policy["identity"]: identity,\n            "archive": str(target),\n            "archiveSha256": archive_sha256,\n            "archiveBytes": target.stat().st_size,\n            "manifestSha256": sha256_bytes(manifest_bytes),\n            "uncompressedInputBytes": total_bytes,\n            "fileCount": len(entries),\n            "manifest": manifest,\n        }\n    except BaseException:\n        staging.unlink(missing_ok=True)\n        raise\n\n\nclass BytesReader:\n    def __init__(self, value: bytes):\n        self.value = value\n        self.offset = 0\n\n    def read(self, size: int = -1) -> bytes:\n        if size < 0:\n            size = len(self.value) - self.offset\n        start = self.offset\n        self.offset = min(len(self.value), self.offset + size)\n        return self.value[start:self.offset]\n\n\ndef regular_tar_info(name: str, size: int) -> tarfile.TarInfo:\n    info = tarfile.TarInfo(name)\n    info.size = size\n    info.mode = 0o644\n    info.uid = 0\n    info.gid = 0\n    info.uname = ""\n    info.gname = ""\n    info.mtime = 0\n    info.type = tarfile.REGTYPE\n    return info\n\n\ndef verify_closed_archive(\n    archive_path: Path,\n    *,\n    expected_kind: str,\n    expected_archive_sha256: str | None = None,\n    extract_to: Path | None = None,\n) -> dict[str, Any]:\n    policy = archive_policy(expected_kind)\n    requested_archive = archive_path.absolute()\n    metadata = requested_archive.lstat()\n    if (\n        stat.S_ISLNK(metadata.st_mode)\n        or not stat.S_ISREG(metadata.st_mode)\n        or not 1 <= metadata.st_size <= MAX_ARCHIVE_BYTES\n    ):\n        raise NeuralRemoteArtifactError("Remote archive is unsafe or too large.")\n    archive = requested_archive.resolve(strict=True)\n    if (\n        expected_archive_sha256 is not None\n        and not is_sha256(expected_archive_sha256)\n    ):\n        raise NeuralRemoteArtifactError("Expected archive SHA-256 is invalid.")\n\n    staging: Path | None = None\n    destination: Path | None = None\n    if extract_to is not None:\n        destination = extract_to.resolve()\n        if destination.exists() or destination.is_symlink():\n            raise NeuralRemoteArtifactError(\n                f"Extraction destination already exists: {destination}"\n            )\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        staging = Path(\n            tempfile.mkdtemp(\n                prefix=f".{destination.name}.staging.",\n                dir=destination.parent,\n            )\n        )\n        os.chmod(staging, 0o700)\n\n    observed: list[dict[str, Any]] = []\n    manifest: dict[str, Any] | None = None\n    manifest_payload: bytes | None = None\n    total_bytes = 0\n    try:\n        with open_regular_binary(archive) as raw:\n            archive_reader = DigestingReader(raw)\n            with tarfile.open(fileobj=archive_reader, mode="r|gz") as tar:\n                saw_manifest = False\n                for member in tar:\n                    validate_tar_member_metadata(member)\n                    expected_prefix = f"{policy[\'prefix\']}/"\n                    if not member.name.startswith(expected_prefix):\n                        raise NeuralRemoteArtifactError(\n                            f"Archive member has the wrong root: {member.name}"\n                        )\n                    relative = safe_relative_path(\n                        member.name[len(expected_prefix):],\n                        "archive member path",\n                    )\n                    stream = tar.extractfile(member)\n                    if stream is None:\n                        raise NeuralRemoteArtifactError(\n                            f"Archive member cannot be read: {relative}"\n                        )\n                    if relative == policy["manifest"]:\n                        if saw_manifest:\n                            raise NeuralRemoteArtifactError(\n                                "Archive contains duplicate manifests."\n                            )\n                        saw_manifest = True\n                        manifest_payload = stream.read(MAX_MANIFEST_BYTES + 1)\n                        if (\n                            len(manifest_payload) > MAX_MANIFEST_BYTES\n                            or len(manifest_payload) != member.size\n                        ):\n                            raise NeuralRemoteArtifactError(\n                                "Archive manifest is truncated or too large."\n                            )\n                        try:\n                            parsed = json.loads(\n                                manifest_payload.decode("utf-8")\n                            )\n                        except (\n                            UnicodeDecodeError,\n                            json.JSONDecodeError,\n                        ) as error:\n                            raise NeuralRemoteArtifactError(\n                                "Archive manifest is invalid JSON."\n                            ) from error\n                        if not isinstance(parsed, dict):\n                            raise NeuralRemoteArtifactError(\n                                "Archive manifest must be an object."\n                            )\n                        manifest = parsed\n                        continue\n                    if saw_manifest:\n                        raise NeuralRemoteArtifactError(\n                            "Archive contains members after its closing manifest."\n                        )\n                    if len(observed) >= MAX_FILES:\n                        raise NeuralRemoteArtifactError(\n                            "Archive contains too many files."\n                        )\n                    if not 1 <= member.size <= MAX_MEMBER_BYTES:\n                        raise NeuralRemoteArtifactError(\n                            f"Archive member size is unsafe: {relative}"\n                        )\n                    total_bytes += member.size\n                    if total_bytes > MAX_TOTAL_BYTES:\n                        raise NeuralRemoteArtifactError(\n                            "Archive exceeds the uncompressed safety limit."\n                        )\n                    digest = hashlib.sha256()\n                    written = 0\n                    output: BinaryIO | None = None\n                    try:\n                        if staging is not None:\n                            output_path = safe_extraction_path(staging, relative)\n                            output_path.parent.mkdir(\n                                parents=True,\n                                exist_ok=True,\n                            )\n                            descriptor = os.open(\n                                output_path,\n                                os.O_WRONLY\n                                | os.O_CREAT\n                                | os.O_EXCL\n                                | getattr(os, "O_NOFOLLOW", 0),\n                                0o600,\n                            )\n                            output = os.fdopen(descriptor, "wb")\n                        while True:\n                            chunk = stream.read(CHUNK_BYTES)\n                            if not chunk:\n                                break\n                            written += len(chunk)\n                            if written > member.size:\n                                raise NeuralRemoteArtifactError(\n                                    f"Archive member exceeds declared size: {relative}"\n                                )\n                            digest.update(chunk)\n                            if output is not None:\n                                output.write(chunk)\n                        if written != member.size:\n                            raise NeuralRemoteArtifactError(\n                                f"Archive member is truncated: {relative}"\n                            )\n                        if output is not None:\n                            output.flush()\n                            os.fsync(output.fileno())\n                            os.fchmod(output.fileno(), 0o644)\n                    finally:\n                        if output is not None:\n                            output.close()\n                    observed.append(\n                        {\n                            "path": relative,\n                            "sha256": digest.hexdigest(),\n                            "bytes": written,\n                        }\n                    )\n            for _chunk in iter(\n                lambda: archive_reader.read(CHUNK_BYTES),\n                b"",\n            ):\n                pass\n        archive_sha256 = archive_reader.hexdigest()\n        metadata_after = archive.lstat()\n        if (\n            archive_reader.bytes_read != metadata.st_size\n            or metadata_after.st_dev != metadata.st_dev\n            or metadata_after.st_ino != metadata.st_ino\n            or metadata_after.st_size != metadata.st_size\n            or metadata_after.st_mtime_ns != metadata.st_mtime_ns\n        ):\n            raise NeuralRemoteArtifactError(\n                "Remote archive changed while it was being verified."\n            )\n        if (\n            expected_archive_sha256 is not None\n            and archive_sha256 != expected_archive_sha256\n        ):\n            raise NeuralRemoteArtifactError(\n                "Remote archive SHA-256 does not match the trusted value."\n            )\n        if manifest is None or manifest_payload is None:\n            raise NeuralRemoteArtifactError("Archive closing manifest is missing.")\n        validate_closed_manifest(\n            manifest,\n            expected_kind=expected_kind,\n            observed=observed,\n        )\n        if staging is not None and destination is not None:\n            manifest_path = staging / policy["manifest"]\n            descriptor = os.open(\n                manifest_path,\n                os.O_WRONLY\n                | os.O_CREAT\n                | os.O_EXCL\n                | getattr(os, "O_NOFOLLOW", 0),\n                0o600,\n            )\n            with os.fdopen(descriptor, "wb") as output:\n                output.write(manifest_payload)\n                output.flush()\n                os.fsync(output.fileno())\n                os.fchmod(output.fileno(), 0o644)\n            os.replace(staging, destination)\n            staging = None\n        return {\n            "schemaVersion": 1,\n            "status": "passed-closed-archive-verification",\n            "artifactKind": expected_kind,\n            policy["identity"]: manifest[policy["identity"]],\n            "archive": str(archive),\n            "archiveSha256": archive_sha256,\n            "archiveBytes": metadata.st_size,\n            "manifestSha256": sha256_bytes(manifest_payload),\n            "fileCount": len(observed),\n            "uncompressedInputBytes": total_bytes,\n            "extractedTo": str(destination) if destination is not None else None,\n            "manifest": manifest,\n        }\n    finally:\n        if staging is not None and staging.exists():\n            shutil.rmtree(staging)\n\n\ndef validate_closed_manifest(\n    manifest: dict[str, Any],\n    *,\n    expected_kind: str,\n    observed: list[dict[str, Any]],\n) -> None:\n    policy = archive_policy(expected_kind)\n    identity_field = policy["identity"]\n    if manifest.get("schemaVersion") != 1:\n        raise NeuralRemoteArtifactError("Archive manifest schema is unsupported.")\n    if manifest.get("artifactKind") != expected_kind:\n        raise NeuralRemoteArtifactError("Archive manifest kind is incorrect.")\n    if manifest.get("archivePrefix") != policy["prefix"]:\n        raise NeuralRemoteArtifactError("Archive prefix attestation is incorrect.")\n    if manifest.get("compression") != "tar-gzip-deterministic-v1":\n        raise NeuralRemoteArtifactError("Archive compression contract is invalid.")\n    identity = manifest.get(identity_field)\n    if not is_sha256(identity) or identity != manifest_identity(\n        manifest,\n        identity_field,\n    ):\n        raise NeuralRemoteArtifactError("Archive manifest identity is invalid.")\n    declared = manifest.get("files")\n    if not isinstance(declared, list) or not declared:\n        raise NeuralRemoteArtifactError("Archive manifest file inventory is empty.")\n    if any(not isinstance(entry, dict) for entry in declared):\n        raise NeuralRemoteArtifactError(\n            "Archive manifest file entry schema is invalid."\n        )\n    if declared != sorted(declared, key=lambda item: item.get("path", "")):\n        raise NeuralRemoteArtifactError("Archive manifest inventory is not sorted.")\n    declared_without_roles: list[dict[str, Any]] = []\n    seen: set[str] = set()\n    for entry in declared:\n        if set(entry) != {\n            "path",\n            "role",\n            "sha256",\n            "bytes",\n        }:\n            raise NeuralRemoteArtifactError(\n                "Archive manifest file entry schema is invalid."\n            )\n        path = safe_relative_path(entry["path"], "manifest file path")\n        if path in seen:\n            raise NeuralRemoteArtifactError(\n                "Archive manifest contains duplicate paths."\n            )\n        seen.add(path)\n        if (\n            not isinstance(entry["role"], str)\n            or not entry["role"]\n            or not is_sha256(entry["sha256"])\n            or type(entry["bytes"]) is not int\n            or not 1 <= entry["bytes"] <= MAX_MEMBER_BYTES\n        ):\n            raise NeuralRemoteArtifactError(\n                f"Archive manifest evidence is invalid: {path}"\n            )\n        declared_without_roles.append(\n            {\n                "path": path,\n                "sha256": entry["sha256"],\n                "bytes": entry["bytes"],\n            }\n        )\n    if declared_without_roles != observed:\n        raise NeuralRemoteArtifactError(\n            "Archive bytes do not match the closed manifest inventory."\n        )\n\n\ndef validate_tar_member_metadata(member: tarfile.TarInfo) -> None:\n    if (\n        not member.isreg()\n        or member.mode != 0o644\n        or member.uid != 0\n        or member.gid != 0\n        or member.uname not in {"", None}\n        or member.gname not in {"", None}\n        or member.mtime != 0\n        or member.linkname\n        or member.pax_headers\n    ):\n        raise NeuralRemoteArtifactError(\n            f"Archive member metadata is unsafe: {member.name}"\n        )\n\n\ndef safe_extraction_path(staging: Path, relative: str) -> Path:\n    target = staging.joinpath(*PurePosixPath(relative).parts)\n    if not target.resolve().is_relative_to(staging.resolve()):\n        raise NeuralRemoteArtifactError(\n            f"Extraction member escapes staging: {relative}"\n        )\n    return target\n\n\ndef verify_extracted_tree(\n    root: Path,\n    *,\n    expected_kind: str,\n    allowed_output_prefixes: Iterable[str] = (),\n) -> dict[str, Any]:\n    policy = archive_policy(expected_kind)\n    requested_root = root.absolute()\n    root_metadata = requested_root.lstat()\n    if (\n        stat.S_ISLNK(root_metadata.st_mode)\n        or not stat.S_ISDIR(root_metadata.st_mode)\n    ):\n        raise NeuralRemoteArtifactError(\n            f"Extracted tree root is unsafe: {requested_root}"\n        )\n    extracted_root = requested_root.resolve(strict=True)\n    manifest_path = contained_regular_file(\n        extracted_root,\n        policy["manifest"],\n    )\n    manifest = read_json_object(manifest_path)\n    declared = manifest.get("files")\n    if not isinstance(declared, list):\n        raise NeuralRemoteArtifactError("Extracted manifest inventory is invalid.")\n    observed = []\n    for entry in declared:\n        if not isinstance(entry, dict):\n            raise NeuralRemoteArtifactError(\n                "Extracted manifest entry is invalid."\n            )\n        relative = safe_relative_path(entry.get("path"), "manifest file path")\n        path = contained_regular_file(extracted_root, relative)\n        size = path.stat().st_size\n        observed.append(\n            {\n                "path": relative,\n                "sha256": sha256_file(path),\n                "bytes": size,\n            }\n        )\n    validate_closed_manifest(\n        manifest,\n        expected_kind=expected_kind,\n        observed=observed,\n    )\n\n    allowed_prefixes = tuple(\n        safe_relative_path(value, "allowed output prefix").rstrip("/") + "/"\n        for value in allowed_output_prefixes\n    )\n    declared_paths = {\n        entry["path"] for entry in declared\n    } | {policy["manifest"]}\n    for current, directories, filenames in os.walk(\n        extracted_root,\n        topdown=True,\n        followlinks=False,\n    ):\n        current_path = Path(current)\n        for name in [*directories, *filenames]:\n            child = current_path / name\n            metadata = child.lstat()\n            if stat.S_ISLNK(metadata.st_mode):\n                raise NeuralRemoteArtifactError(\n                    f"Extracted tree contains a symbolic link: {child}"\n                )\n            if not (\n                stat.S_ISDIR(metadata.st_mode)\n                or stat.S_ISREG(metadata.st_mode)\n            ):\n                raise NeuralRemoteArtifactError(\n                    f"Extracted tree contains a special file: {child}"\n                )\n        for filename in filenames:\n            path = current_path / filename\n            relative = path.relative_to(extracted_root).as_posix()\n            if relative in declared_paths:\n                continue\n            if any(relative.startswith(prefix) for prefix in allowed_prefixes):\n                continue\n            raise NeuralRemoteArtifactError(\n                f"Extracted tree contains an unlisted file: {relative}"\n            )\n    return manifest\n\n\ndef archive_policy(artifact_kind: str) -> dict[str, str]:\n    try:\n        return ARCHIVE_POLICIES[artifact_kind]\n    except KeyError as error:\n        raise NeuralRemoteArtifactError(\n            f"Unsupported closed archive kind: {artifact_kind}"\n        ) from error\n'

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_expected_archive(path, label):
    if path.is_symlink():
        raise RuntimeError(f"{label} must not be a symbolic link.")
    if not path.exists() or not path.is_file():
        raise RuntimeError(f"{label} is not a regular file.")
    if path.stat().st_size != EXPECTED_ARCHIVE_BYTES:
        raise RuntimeError(f"{label} byte count is wrong.")
    observed = file_sha256(path)
    if observed != EXPECTED_ARCHIVE_SHA256:
        raise RuntimeError(f"{label} SHA-256 is wrong.")
    return observed

def copy_verified(source, target, label):
    staging = target.with_name(
        f".{target.name}.staging.{uuid.uuid4().hex}"
    )
    try:
        if staging.exists() or staging.is_symlink():
            raise RuntimeError(f"{label} staging path unexpectedly exists.")
        shutil.copyfile(source, staging)
        verify_expected_archive(staging, f"{label} staging copy")
        os.replace(staging, target)
        verify_expected_archive(target, label)
    finally:
        if staging.exists() and not staging.is_symlink():
            staging.unlink()

drive.mount("/content/drive")
persistent_base = Path("/content/drive/MyDrive/Lekh-Neural-Training")
persistent_base.mkdir(parents=True, exist_ok=True)
if persistent_base.is_symlink() or not persistent_base.is_dir():
    raise RuntimeError("Durable training root is not a safe directory.")

archive = Path("/content") / EXPECTED_ARCHIVE_NAME
drive_archive = persistent_base / EXPECTED_ARCHIVE_NAME
if archive.exists() or archive.is_symlink():
    verify_expected_archive(archive, "Existing session archive")
    archive_source = "verified-session-cache"
elif drive_archive.exists() or drive_archive.is_symlink():
    verify_expected_archive(drive_archive, "Durable Drive archive")
    copy_verified(
        drive_archive,
        archive,
        "Restored session archive",
    )
    archive_source = "verified-drive-recovery"
else:
    uploaded = files.upload()
    if set(uploaded) != {EXPECTED_ARCHIVE_NAME}:
        raise RuntimeError(
            f"Upload exactly {EXPECTED_ARCHIVE_NAME}; "
            f"observed {sorted(uploaded)}"
        )
    payload = uploaded[EXPECTED_ARCHIVE_NAME]
    if len(payload) != EXPECTED_ARCHIVE_BYTES:
        raise RuntimeError("Uploaded archive byte count is wrong.")
    staging = archive.with_name(
        f".{archive.name}.staging.{uuid.uuid4().hex}"
    )
    try:
        with staging.open("xb") as output:
            output.write(payload)
            output.flush()
            os.fsync(output.fileno())
        verify_expected_archive(staging, "Uploaded staging archive")
        os.replace(staging, archive)
    finally:
        if staging.exists() and not staging.is_symlink():
            staging.unlink()
    verify_expected_archive(archive, "Uploaded archive")
    archive_source = "verified-browser-upload"

if drive_archive.exists() or drive_archive.is_symlink():
    verify_expected_archive(drive_archive, "Durable Drive archive")
else:
    copy_verified(
        archive,
        drive_archive,
        "Durable Drive archive",
    )

bootstrap = Path("/content/lekh-neural-bootstrap")
bootstrap.mkdir(mode=0o700, exist_ok=True)
if bootstrap.is_symlink() or not bootstrap.is_dir():
    raise RuntimeError("Verifier bootstrap root is not a safe directory.")
module_path = bootstrap / "neural_remote_artifacts.py"
verifier_payload = VERIFIER_MODULE_SOURCE.encode("utf-8")
if module_path.exists() or module_path.is_symlink():
    if (
        module_path.is_symlink()
        or not module_path.is_file()
        or module_path.read_bytes() != verifier_payload
    ):
        raise RuntimeError("Existing verifier module differs from this notebook.")
else:
    with module_path.open("xb") as output:
        output.write(verifier_payload)
        output.flush()
        os.fsync(output.fileno())
spec = importlib.util.spec_from_file_location(
    "neural_remote_artifacts",
    module_path,
)
if spec is None or spec.loader is None:
    raise RuntimeError("Unable to load the embedded archive verifier.")
remote_artifacts = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = remote_artifacts
try:
    spec.loader.exec_module(remote_artifacts)
except Exception:
    sys.modules.pop(spec.name, None)
    raise

bundle_root = Path("/content/lekh-neural-remote")
if bundle_root.exists():
    manifest = remote_artifacts.verify_extracted_tree(
        bundle_root,
        expected_kind=remote_artifacts.BUNDLE_KIND,
        allowed_output_prefixes=(EXPECTED_CANDIDATE_PREFIX,),
    )
    if manifest["bundleId"] != EXPECTED_BUNDLE_ID:
        raise RuntimeError("Existing extraction belongs to another bundle.")
else:
    verification = remote_artifacts.verify_closed_archive(
        archive,
        expected_kind=remote_artifacts.BUNDLE_KIND,
        expected_archive_sha256=EXPECTED_ARCHIVE_SHA256,
        extract_to=bundle_root,
    )
    if verification["bundleId"] != EXPECTED_BUNDLE_ID:
        raise RuntimeError("Verified archive has an unexpected bundleId.")

print(json.dumps({
    "status": "passed-drive-first-closed-inventory-verification",
    "bundleId": EXPECTED_BUNDLE_ID,
    "modelId": EXPECTED_MODEL_ID,
    "archiveSha256": EXPECTED_ARCHIVE_SHA256,
    "archiveSource": archive_source,
    "durableArchive": str(drive_archive),
}, indent=2))

## 2. Inspect durable progress

This lightweight cell reports the observed completed epoch or final-result pointer without requiring a GPU. It does not replace the runner's full authenticated recovery check.

In [ ]:
def read_optional_pointer(path, label):
    if path.is_symlink():
        raise RuntimeError(f"{label} must not be a symbolic link.")
    if not path.exists():
        return None
    if not path.is_file():
        raise RuntimeError(f"{label} is not a regular file.")
    if not 1 <= path.stat().st_size <= 64 * 1024:
        raise RuntimeError(f"{label} is empty or unexpectedly large.")
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise RuntimeError(f"{label} must contain one JSON object.")
    return value

remote_root = (
    persistent_base
    / "lekh-neural-remote"
    / EXPECTED_BUNDLE_ID
    / EXPECTED_MODEL_ID
)
recovery_root = (
    remote_root
    / "recovery"
    / EXPECTED_BUNDLE_ID
    / EXPECTED_MODEL_ID
)
result_root = remote_root / "results"
recovery_pointer = read_optional_pointer(
    recovery_root / "LATEST.json",
    "Recovery pointer",
)
result_pointer = read_optional_pointer(
    result_root / "LATEST_RESULT.json",
    "Result pointer",
)

status = {
    "schemaVersion": 1,
    "bundleId": EXPECTED_BUNDLE_ID,
    "modelId": EXPECTED_MODEL_ID,
    "recovery": None,
    "result": None,
    "note": (
        "This is a lightweight status view. The runner authenticates the "
        "complete recovery generation before resuming."
    ),
}
if recovery_pointer is not None:
    generation = recovery_pointer.get("generation")
    recovery_id = recovery_pointer.get("recoveryId")
    completed_epoch = recovery_pointer.get("completedEpoch")
    if (
        set(recovery_pointer) != {
            "schemaVersion",
            "bundleId",
            "modelId",
            "generation",
            "recoveryId",
            "completedEpoch",
        }
        or recovery_pointer.get("schemaVersion") != 1
        or recovery_pointer.get("bundleId") != EXPECTED_BUNDLE_ID
        or recovery_pointer.get("modelId") != EXPECTED_MODEL_ID
        or not isinstance(generation, str)
        or re.fullmatch(r"epoch-[0-9]{6}-[0-9a-f]{16}", generation) is None
        or not isinstance(recovery_id, str)
        or re.fullmatch(r"[0-9a-f]{64}", recovery_id) is None
        or type(completed_epoch) is not int
        or completed_epoch < 1
    ):
        raise RuntimeError("Recovery pointer identity is malformed or stale.")
    status["recovery"] = {
        "status": "observed-recoverable-pointer",
        "completedEpoch": completed_epoch,
        "generation": generation,
        "recoveryId": recovery_id,
    }
if result_pointer is not None:
    archive_name = result_pointer.get("archive")
    archive_sha256 = result_pointer.get("archiveSha256")
    archive_bytes = result_pointer.get("archiveBytes")
    training_run_id = result_pointer.get("trainingRunId")
    result_id = result_pointer.get("resultId")
    if (
        set(result_pointer) != {
            "schemaVersion",
            "status",
            "bundleId",
            "modelId",
            "trainingRunId",
            "resultId",
            "archive",
            "archiveSha256",
            "archiveBytes",
        }
        or result_pointer.get("schemaVersion") != 1
        or result_pointer.get("status") != "complete-neural-remote-result"
        or result_pointer.get("bundleId") != EXPECTED_BUNDLE_ID
        or result_pointer.get("modelId") != EXPECTED_MODEL_ID
        or not isinstance(training_run_id, str)
        or re.fullmatch(r"[0-9a-f]{32}", training_run_id) is None
        or not isinstance(result_id, str)
        or re.fullmatch(r"[0-9a-f]{64}", result_id) is None
        or not isinstance(archive_name, str)
        or Path(archive_name).name != archive_name
        or re.fullmatch(
            r"[A-Za-z0-9][A-Za-z0-9._-]{0,179}[.]tar[.]gz",
            archive_name,
        ) is None
        or not isinstance(archive_sha256, str)
        or re.fullmatch(r"[0-9a-f]{64}", archive_sha256) is None
        or type(archive_bytes) is not int
        or archive_bytes < 1
    ):
        raise RuntimeError("Result pointer identity is malformed or stale.")
    status["result"] = {
        "status": "observed-complete-result-pointer",
        "archive": archive_name,
        "archiveSha256": archive_sha256,
        "archiveBytes": archive_bytes,
    }
print(json.dumps(status, indent=2))

## 3. Verify GPU and install the pinned Python toolchain

The first check fails immediately on a CPU-only runtime, before downloading the training toolchain.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

UV_VERSION = '0.11.8'
PINNED_PYTHON = '3.11.15'
REMOTE_TORCH = "torch==2.7.0+cu118"
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu118"
venv = Path("/content/lekh-neural-venv-py31115")

system_nvidia_smi = shutil.which("nvidia-smi")
if system_nvidia_smi is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime is attached. Use Runtime → Change runtime "
        "type → GPU, or wait for the free GPU quota to reset."
    )
gpu_preflight = subprocess.run(
    [
        system_nvidia_smi,
        "--query-gpu=name",
        "--format=csv,noheader",
    ],
    check=False,
    capture_output=True,
    text=True,
)
if gpu_preflight.returncode != 0 or not gpu_preflight.stdout.strip():
    raise RuntimeError(
        "The NVIDIA GPU runtime is not usable: "
        f"{gpu_preflight.stderr.strip()}"
    )
print(json.dumps({
    "status": "passed-early-gpu-preflight",
    "devices": gpu_preflight.stdout.strip().splitlines(),
}, indent=2))

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        f"uv=={UV_VERSION}",
    ],
    check=True,
)
uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("Pinned uv executable was not installed.")
subprocess.run([uv, "python", "install", PINNED_PYTHON], check=True)
if not venv.exists():
    subprocess.run(
        [
            uv,
            "venv",
            "--seed",
            "--python",
            PINNED_PYTHON,
            str(venv),
        ],
        check=True,
    )

python = venv / "bin/python"
lock_path = bundle_root / "requirements/neural-open-vocab.lock"
cuda_lock_path = (
    bundle_root / "requirements/neural-open-vocab-cu118.lock"
)
locked_requirements = [
    line.strip()
    for line in lock_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
torch_requirements = [
    value for value in locked_requirements
    if value.startswith("torch==")
]
if torch_requirements != ["torch==2.7.0"]:
    raise RuntimeError(
        f"Unexpected base torch lock: {torch_requirements}"
    )
non_torch_requirements = [
    value for value in locked_requirements
    if not value.startswith("torch==")
]
cuda_requirements = [
    line.strip()
    for line in cuda_lock_path.read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
cuda_torch_requirements = [
    value for value in cuda_requirements
    if value.startswith("torch==")
]
if cuda_torch_requirements != [REMOTE_TORCH]:
    raise RuntimeError(
        f"Unexpected CUDA torch lock: {cuda_torch_requirements}"
    )
subprocess.run(
    [
        uv,
        "pip",
        "install",
        "--python",
        str(python),
        "--no-deps",
        *non_torch_requirements,
    ],
    check=True,
)
subprocess.run(
    [
        uv,
        "pip",
        "install",
        "--python",
        str(python),
        "--no-deps",
        "--index-url",
        PYTORCH_INDEX,
        *cuda_requirements,
    ],
    check=True,
)
subprocess.run(
    [str(python), "-m", "pip", "check"],
    check=True,
)
python_version = subprocess.run(
    [
        str(python),
        "-c",
        "import platform; print(platform.python_version())",
    ],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if python_version != PINNED_PYTHON:
    raise RuntimeError(
        f"Remote Python must be {PINNED_PYTHON}; observed {python_version}."
    )
toolchain = subprocess.run(
    [
        str(python),
        str(bundle_root / "scripts/check-neural-open-vocab-toolchain.py"),
        "--profile",
        "linux-cuda-cu118",
    ],
    cwd=bundle_root,
    check=True,
    capture_output=True,
    text=True,
)
print(toolchain.stdout)
cuda = subprocess.run(
    [
        str(python),
        "-c",
        (
            "import json, torch; "
            "print(json.dumps({'available':torch.cuda.is_available(),"
            "'device':torch.cuda.get_device_name(0) if "
            "torch.cuda.is_available() else None,"
            "'torch':torch.__version__,'cuda':torch.version.cuda},indent=2))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
cuda_report = json.loads(cuda.stdout)
if cuda_report["available"] is not True:
    raise RuntimeError(
        "CUDA is unavailable. In Colab choose Runtime → Change runtime type → GPU."
    )
if (
    cuda_report["torch"] != "2.7.0+cu118"
    or cuda_report["cuda"] != "11.8"
):
    raise RuntimeError(
        f"CUDA runtime drifted from the pinned cu118 profile: {cuda_report}"
    )
print(json.dumps(cuda_report, indent=2))

## 4. Train or resume

Re-running this cell resumes the newest fully mirrored epoch when the runtime fingerprint is compatible. If Colab assigns different CUDA hardware, the trainer fails closed instead of pretending the continuation is bit-reproducible.

In [ ]:
import os
import subprocess

training_environment = os.environ.copy()
training_environment["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
training_environment["PYTHONHASHSEED"] = "42"
training_environment["PYTHONDONTWRITEBYTECODE"] = "1"
training_environment["PYTHONUNBUFFERED"] = "1"
subprocess.run(
    [
        str(python),
        "-u",
        "-B",
        str(bundle_root / "scripts/run-neural-remote-training.py"),
        "--config",
        EXPECTED_CONFIG,
        "--persistent-dir",
        str(persistent_base),
    ],
    cwd=bundle_root,
    env=training_environment,
    check=True,
)

## 5. Download the authenticated training result

This result contains only the checkpoint, vocabulary, and training evidence. Core ML conversion remains a separate short macOS operation.

In [ ]:
from google.colab import files
from pathlib import Path
import json
import re

result_root = (
    persistent_base
    / "lekh-neural-remote"
    / EXPECTED_BUNDLE_ID
    / EXPECTED_MODEL_ID
    / "results"
)
latest_path = result_root / "LATEST_RESULT.json"
if (
    latest_path.is_symlink()
    or not latest_path.is_file()
    or not 1 <= latest_path.stat().st_size <= 64 * 1024
):
    raise RuntimeError("Remote result pointer is missing or unsafe.")
latest = json.loads(latest_path.read_text(encoding="utf-8"))
if not isinstance(latest, dict):
    raise RuntimeError("Remote result pointer must contain one JSON object.")
archive_name = latest.get("archive")
if (
    set(latest) != {
        "schemaVersion",
        "status",
        "bundleId",
        "modelId",
        "trainingRunId",
        "resultId",
        "archive",
        "archiveSha256",
        "archiveBytes",
    }
    or latest.get("schemaVersion") != 1
    or latest.get("status") != "complete-neural-remote-result"
    or latest.get("bundleId") != EXPECTED_BUNDLE_ID
    or latest.get("modelId") != EXPECTED_MODEL_ID
    or not isinstance(latest.get("trainingRunId"), str)
    or re.fullmatch(r"[0-9a-f]{32}", latest["trainingRunId"]) is None
    or not isinstance(latest.get("resultId"), str)
    or re.fullmatch(r"[0-9a-f]{64}", latest["resultId"]) is None
    or not isinstance(archive_name, str)
    or Path(archive_name).name != archive_name
    or re.fullmatch(
        r"[A-Za-z0-9][A-Za-z0-9._-]{0,179}[.]tar[.]gz",
        archive_name,
    ) is None
    or not isinstance(latest.get("archiveSha256"), str)
    or re.fullmatch(r"[0-9a-f]{64}", latest["archiveSha256"]) is None
    or type(latest.get("archiveBytes")) is not int
    or latest["archiveBytes"] < 1
):
    raise RuntimeError("Remote result pointer identity is malformed or stale.")
result_archive = result_root / archive_name
if result_archive.is_symlink() or not result_archive.is_file():
    raise RuntimeError("Remote result archive is missing or unsafe.")
if result_archive.stat().st_size != latest["archiveBytes"]:
    raise RuntimeError("Remote result archive byte count is stale.")
if file_sha256(result_archive) != latest["archiveSha256"]:
    raise RuntimeError("Remote result archive SHA-256 is stale.")
print(json.dumps(latest, indent=2))
files.download(str(result_archive))